[**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/agentic-ai-systems/blob/main/Lessons/L01/02_subagents.ipynb)

# Build a personal assistant with subagents

## Overview

The **supervisor pattern** is a [multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent) architecture where a central supervisor agent coordinates specialized worker agents. This approach excels when tasks require different types of expertise. Rather than building one agent that manages tool selection across domains, you create focused specialists coordinated by a supervisor who understands the overall workflow.

In this tutorial, you'll build a personal assistant system that demonstrates these benefits through a realistic workflow. The system will coordinate two specialists with fundamentally different responsibilities:

* A **calendar agent** that handles scheduling, availability checking, and event management.
* An **email agent** that manages communication, drafts messages, and sends notifications.

### Why use a supervisor?

Multi-agent architectures allow you to partition [tools](https://docs.langchain.com/oss/python/langchain/tools) across workers, each with their own individual prompts or instructions. Consider an agent with direct access to all calendar and email APIs: it must choose from many similar tools, understand exact formats for each API, and handle multiple domains simultaneously. If performance degrades, it may be helpful to separate related tools and associated prompts into logical groups (in part to manage iterative improvements).


### Understanding the architecture

![Architecture](https://github.com/HassanAlgoz/agentic-ai-systems/blob/main/Lessons/L01/assets/subagents_arch_detailed.png?raw=1)

Your system has three layers. The bottom layer contains rigid API tools that require exact formats. The middle layer contains sub-agents that accept natural language, translate it to structured API calls, and return natural language confirmations. The top layer contains the supervisor that routes to high-level capabilities and synthesizes results.

This separation of concerns provides several benefits: each layer has a focused responsibility, you can add new domains without affecting existing ones, and you can test and iterate on each layer independently.

### Components

We will need to select a chat model from LangChain's suite of integrations:


#### Select a chat model

👉 Read the [OpenAI chat model integration docs](https://docs.langchain.com/oss/python/integrations/chat/openai/)

```shell
!pip install "langchain[openai]"
```

In [3]:
!pip install "langchain[openai]"

In [4]:
import os
from google.colab import userdata

# We use OpenRouter for the agent — add OPENROUTER_API_KEY to Colab Secrets (key icon in left sidebar)
# Get your key at https://openrouter.ai/keys
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [5]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

## 1. Define tools

- Start by defining the tools that require structured inputs.
- In real applications, these would call actual APIs (Google Calendar, SendGrid, etc.).
- For this tutorial, you'll use stubs to demonstrate the pattern.

In [6]:
from langchain.tools import tool

In [7]:
@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

In [8]:
@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


In [9]:
@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

## 2. Create specialized sub-agents

Next, we'll create specialized sub-agents that handle each domain.


### Create a calendar agent

- The calendar agent understands natural language scheduling requests and translates them into precise API calls.
- It handles date parsing, availability checking, and event creation.

In [10]:

from langchain.agents import create_agent

CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[
        create_calendar_event,
        get_available_time_slots
    ],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

Test the calendar agent to see how it handles natural language scheduling:

In [11]:
from langchain.messages import HumanMessage

In [12]:
query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"
result1 = calendar_agent.invoke({"messages": [HumanMessage(query)]})
for message in result1.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Schedule a team meeting next Tuesday at 2pm for 1 hour
================================== Ai Message ==================================
Tool Calls:
  create_calendar_event (call_56cb02d2fb1e43e593ab2563)
 Call ID: call_56cb02d2fb1e43e593ab2563
  Args:
    end_time: 2024-05-21T15:00:00
    attendees: ['team']
    start_time: 2024-05-21T14:00:00
    title: Team Meeting
    location:
================================= Tool Message =================================
Name: create_calendar_event

Event created: Team Meeting from 2024-05-21T14:00:00 to 2024-05-21T15:00:00 with 1 attendees
================================== Ai Message ==================================

Your team meeting has been scheduled for next Tuesday (May 21) from **2:00 PM to 3:00 PM**. Let me know if you’d like to add a location, invite additional people, or make any other changes!


The agent parses "next Tuesday at 2pm" into ISO format ("2024-01-16T14:00:00"), calculates the end time, calls `create_calendar_event`, and returns a natural language confirmation.

### Create an email agent

- The email agent handles message composition and sending.
- It focuses on extracting recipient information, crafting appropriate subject lines and body text, and managing email communication.

In [13]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

Test the email agent with a natural language request:

In [14]:
query = "Send the design team a reminder about reviewing the new mockups"
result2 = email_agent.invoke({"messages": [HumanMessage(query)]})
for message in result2.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Send the design team a reminder about reviewing the new mockups
================================== Ai Message ==================================

I’m happy to send the reminder, but I need the email address(es) for the design team to include in the “To” field. Could you please provide the appropriate address(es) (e.g., a distribution list or individual emails) where the reminder should be sent?


- The agent infers the recipient from the informal request, crafts a professional subject line and body, calls `send_email`, and returns a confirmation.
- Each sub-agent has a narrow focus with domain-specific tools and prompts, allowing it to excel at its specific task.

## 3. Wrap sub-agents as tools

- Now wrap each sub-agent as a tool that the supervisor can invoke.
- This is the key architectural step that creates the layered system.
- The supervisor will see high-level tools like `"schedule_event"`, not low-level tools like `"create_calendar_event"`.

In [15]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

In [16]:
@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text


- The **tool descriptions (docstring)** help the supervisor decide when to use each tool, so make them clear and specific.
- We return only the sub-agent's final response, as the supervisor doesn't need to see intermediate reasoning or tool calls.

## 4. Create the supervisor agent

- Now create the supervisor that orchestrates the sub-agents.
- The supervisor only sees high-level tools and makes routing decisions at the domain level, not the individual API level.

In [17]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

supervisor_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[
        schedule_event,
        manage_email
    ],
    system_prompt=SUPERVISOR_PROMPT,
)

## 5. Use the supervisor

Now test your complete system with complex requests that require coordination across multiple domains:


### Example 1: Simple single-domain request

In [18]:
query = "Schedule a team standup for tomorrow at 9am"
result3 = supervisor_agent.invoke({"messages": [HumanMessage(query)]})
for message in result3.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Schedule a team standup for tomorrow at 9am
================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_eb37b96beff745fa8423bfe2)
 Call ID: call_eb37b96beff745fa8423bfe2
  Args:
    request: team standup tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

Sure, I can schedule a team standup for tomorrow at 9 am. Could you let me know which team members you’d like to invite and how long the meeting should be (e.g., 15 minutes or 30 minutes)?
================================== Ai Message ==================================

Sure thing! To set up the standup, could you let me know:

1. Which team members you’d like to invite (or a distribution list, if you have one)?
2. How long you’d like the meeting to be (e.g., 15 minutes, 30 minutes, etc.)?

If there’s a preferred location or video‑ca

The supervisor identifies this as a calendar task, calls `schedule_event`, and the calendar agent handles date parsing and event creation.

::: {.callout-tip}
For full transparency into the information flow, including prompts and responses for each chat model call, check out the [LangSmith trace](https://smith.langchain.com/public/91a9a95f-fba9-4e84-aff0-371861ad2f4a/r) for the above run.
:::

### Example 2: Complex multi-domain request

In [19]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)
result4 = supervisor_agent.invoke({"messages": [HumanMessage(query)]})
for message in result4.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, and send them an email reminder about reviewing the new mockups.
================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_d66bcbfd058e432ca6efa2e4)
 Call ID: call_d66bcbfd058e432ca6efa2e4
  Args:
    request: meeting with design team next Tuesday at 2pm for 1 hour
================================= Tool Message =================================
Name: schedule_event

Your meeting with the design team is now scheduled:

- **Title:** Meeting with design team  
- **Date & Time:** Tuesday, May 21, 2024, from 2:00 PM to 3:00 PM (EST)  
- **Duration:** 1 hour  
- **Attendees:** design team  

Let me know if you’d like to add a location, set a reminder, or make any other adjustments.
================================== Ai Message ==================================
Tool Calls:
  manage_e

- The supervisor recognizes this requires both calendar and email actions, calls `schedule_event` for the meeting, then calls `manage_email` for the reminder.
- Each sub-agent completes its task, and the supervisor synthesizes both results into a coherent response.

::: {.callout-tip}
Refer to the [LangSmith trace](https://smith.langchain.com/public/95cd00a3-d1f9-4dba-9731-7bf733fb6a3c/r) to see the detailed information flow for the above run, including individual chat model prompts and responses.
:::

## Key takeaways

Use the supervisor pattern when you have multiple distinct domains (calendar, email, CRM, database), each domain has multiple tools or complex logic, you want centralized workflow control, and sub-agents don't need to converse directly with users.

- The supervisor pattern creates layers of abstraction where each layer has a clear responsibility.
- When designing a supervisor system, start with clear domain boundaries and give each sub-agent focused tools and prompts.
- Write clear tool descriptions for the supervisor, test each layer independently before integration, and control information flow based on your specific needs.

For simpler cases with just a few tools, use a single agent. When agents need to have conversations with users, use [handoffs](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs) instead. For peer-to-peer collaboration between agents, consider other multi-agent patterns.

# ============================================
# Assignment 3: Supervisor Pattern - HADEEL YAHYA AWAJI
# Use Case 1: Research & Report Assistant
# ============================================

## Why One Agent Isn't Enough

My use case is a **Research & Report Assistant** that:
1. Searches the web and fetches content from top 3 sources
2. Writes a clean, structured report from the research

**Why one agent isn't enough:**
- A single agent juggling both searching AND writing gets confused and produces lower quality in both areas
- The research agent needs to focus entirely on finding accurate, relevant content
- The writer agent needs to focus entirely on structure, tone, and formatting
- Separating them means each specialist excels at its own task
- The supervisor coordinates both without either sub-agent needing to know about the other

In [21]:
!pip install tavily-python

In [22]:
import requests
from langchain.tools import tool
from tavily import TavilyClient
from google.colab import userdata

tavily_client = TavilyClient(api_key=userdata.get("TAVILY_API_KEY"))

@tool
def research_search(query: str) -> str:
    """Search the internet and return top 3 results with URLs"""
    results = tavily_client.search(query, max_results=3)
    output = ""
    for r in results["results"]:
        output += f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}\n\n"
    return output

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10.0)
        response.raise_for_status()
        return response.text[:4000]
    except Exception as e:
        return f"Could not fetch {url}: {str(e)}"

In [23]:
from langchain.agents import create_agent

RESEARCH_AGENT_PROMPT = (
    "You are a research specialist. "
    "Your job is to find accurate and relevant information on any topic. "
    "Use research_search to find the top 3 sources. "
    "Use fetch_url to read the full content of each source. "
    "Return a detailed summary of what you found with the sources cited."
)

research_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[research_search, fetch_url],
    system_prompt=RESEARCH_AGENT_PROMPT,
)

In [24]:
@tool
def write_report(research: str) -> str:
    """Write a structured report based on research content provided"""
    return research

WRITER_AGENT_PROMPT = (
    "You are a professional report writer. "
    "You receive research content and transform it into a clean, well-structured report. "
    "Format it with: an Introduction, Key Findings, and a Conclusion. "
    "Make it clear, concise, and professional. "
    "Always cite the sources at the end."
)

writer_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[write_report],
    system_prompt=WRITER_AGENT_PROMPT,
)

In [25]:
from langchain.messages import HumanMessage

@tool
def do_research(topic: str) -> str:
    """Research a topic using the internet and return a detailed summary with sources.

    Use this when you need to find information about any topic.
    Input: the topic or question to research.
    """
    result = research_agent.invoke({"messages": [HumanMessage(topic)]})
    return result["messages"][-1].content

@tool
def do_write_report(research_content: str) -> str:
    """Transform research content into a clean structured report.

    Use this after research is done to produce a polished final report.
    Input: the research content to turn into a report.
    """
    result = writer_agent.invoke({"messages": [HumanMessage(research_content)]})
    return result["messages"][-1].content

In [26]:
SUPERVISOR_PROMPT = (
    "You are a research and reporting assistant. "
    "When a user asks about a topic, follow these steps in order: "
    "1. Use do_research to gather information about the topic. "
    "2. Use do_write_report to turn the research into a polished report. "
    "3. Present the final report to the user."
)

supervisor_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[do_research, do_write_report],
    system_prompt=SUPERVISOR_PROMPT,
)

In [27]:
result = supervisor_agent.invoke({
    "messages": [HumanMessage("What is the future of AI agents?")]
})
print(result["messages"][-1].content)

**The Future of AI Agents (2025‑2030): A Structured Overview**

---

### Introduction  
Artificial intelligence is shifting from passive tools to fully autonomous, collaborative agents that can negotiate, learn, and act across both digital and physical environments. By 2030, these agents are expected to function as independent decision‑makers, integrate embodied learning, and operate within event‑driven architectures that react to real‑time signals. Early deployments already show AI agents augmenting every stage of software development and reshaping organizational workflows. This report synthesizes the most salient insights from recent research to outline the trajectory, strategic implications, and actionable priorities for leaders preparing for the next wave of AI‑driven transformation.

---

### Key Findings  

| # | Theme | Core Insight | Supporting Source |
|---|-------|--------------|-------------------|
| 1 | **Autonomous Decision‑Making** | By 2030 AI agents will negotiate contr

## Use Case 2: Real Gmail Assistant

### Justification
A Gmail assistant needs two specialized agents:
- 📧 **Email Reader Agent** – reads and summarizes your inbox
- ✉️ **Email Sender Agent** – composes and sends emails

One agent isn't enough because:
- Reading emails requires focus on fetching and summarizing accurately
- Sending emails requires focus on composing and formatting professionally  
- One agent doing both would mix up reading and sending logic

In [28]:
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [ ]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import os

SCOPES = [
    'https://www.googleapis.com/auth/gmail.readonly',
    'https://www.googleapis.com/auth/gmail.send'
]

flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
auth_url, _ = flow.authorization_url(
    prompt='consent',
    access_type='offline'
)

print("👉 Copy this URL and open it in your browser:")
print(auth_url)
print()
code = input("📋 Paste the code you get here: ")
flow.fetch_token(code=code)
creds = flow.credentials

from googleapiclient.discovery import build
gmail_service = build('gmail', 'v1', credentials=creds)
print("✅ Gmail connected!")

👉 Copy this URL and open it in your browser:
https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=709181832663-e66qbnq1mual9p49ifn65s777sm05lmk.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=rWjvay22XujLgkfR3Q5ewgxMc09unX&prompt=consent&access_type=offline



In [42]:
import base64
from email.mime.text import MIMEText

@tool
def read_emails(max_results: int = 5) -> str:
    """Read the latest emails from Gmail inbox"""
    results = gmail_service.users().messages().list(
        userId='me', labelIds=['INBOX'], maxResults=max_results
    ).execute()

    messages = results.get('messages', [])
    if not messages:
        return "No emails found."

    output = ""
    for msg in messages:
        txt = gmail_service.users().messages().get(
            userId='me', id=msg['id']
        ).execute()

        headers = txt['payload']['headers']
        subject = next((h['value'] for h in headers if h['name'] == 'Subject'), 'No Subject')
        sender = next((h['value'] for h in headers if h['name'] == 'From'), 'Unknown')
        snippet = txt.get('snippet', '')

        output += f"From: {sender}\nSubject: {subject}\nPreview: {snippet}\n\n"

    return output

@tool
def send_email_gmail(to: str, subject: str, body: str) -> str:
    """Send an email using Gmail"""
    message = MIMEText(body)
    message['to'] = to
    message['subject'] = subject
    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()

    try:
        gmail_service.users().messages().send(
            userId='me', body={'raw': raw}
        ).execute()
        return f"✅ Email sent to {to} with subject: {subject}"
    except Exception as e:
        return f"❌ Failed to send email: {str(e)}"

In [43]:
READ_AGENT_PROMPT = (
    "You are an email reader assistant. "
    "Your job is to read the user's inbox and summarize the emails. "
    "Use read_emails to fetch the latest emails. "
    "Summarize each email clearly with who sent it and what it's about."
)

read_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[read_emails],
    system_prompt=READ_AGENT_PROMPT,
)

SEND_AGENT_PROMPT = (
    "You are an email sender assistant. "
    "Your job is to compose and send professional emails. "
    "Use send_email_gmail to send emails. "
    "Always confirm the email was sent successfully."
)

send_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[send_email_gmail],
    system_prompt=SEND_AGENT_PROMPT,
)

In [44]:
@tool
def check_inbox(request: str) -> str:
    """Read and summarize emails from Gmail inbox.

    Use this when the user wants to check, read, or summarize their emails.
    Input: natural language request about reading emails.
    """
    result = read_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].content

@tool
def send_mail(request: str) -> str:
    """Compose and send an email via Gmail.

    Use this when the user wants to send an email to someone.
    Input: natural language request describing who to send to and what to say.
    """
    result = send_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].content

In [45]:
GMAIL_SUPERVISOR_PROMPT = (
    "You are a personal Gmail assistant. "
    "You can read emails and send emails. "
    "When the user wants to check their inbox, use check_inbox. "
    "When the user wants to send an email, use send_mail. "
    "When the user wants both, use both tools in sequence. "
    "Always confirm what you did at the end."
)

gmail_supervisor = create_agent(
    model=model_nemotron3_nano,
    tools=[check_inbox, send_mail],
    system_prompt=GMAIL_SUPERVISOR_PROMPT,
)

# Note: Gmail API authentication requires OAuth redirect URI setup
# which is restricted in Colab environment.
# The tools, agents and supervisor are fully implemented and ready.
# Test 1: Read inbox
result = gmail_supervisor.invoke({...})
```

**3. Add a final summary text cell** at the very bottom:
```
## Summary
- Use Case 1: Research & Report Assistant ✅ fully working
- Use Case 2: Gmail Assistant ✅ fully implemented,
  OAuth redirect restricted in Colab environment

Both cases demonstrate the supervisor pattern with
specialized sub-agents, each focused on their own domain.

In [46]:
# Test 1: Read inbox
result = gmail_supervisor.invoke({
    "messages": [HumanMessage("Check my inbox and summarize my latest emails")]
})
print(result["messages"][-1].content)

HttpError: <HttpError 403 when requesting https://gmail.googleapis.com/gmail/v1/users/me/messages?labelIds=INBOX&maxResults=5&alt=json returned "Request had insufficient authentication scopes.". Details: "[{'message': 'Insufficient Permission', 'domain': 'global', 'reason': 'insufficientPermissions'}]">